In [1]:
import torch
import numpy as np

from dataclasses import dataclass, field

from tqdm import tqdm

from env.wrapper import EnvWrapper
from RL.models.policy import PolicyNetwork
#from RL.ppo.train_BLpolicy import PPOConfig#, model_summary

from game.enums import BoardType, Tribes, ActionTypes

from RL.models.policy import PolicyNetwork, model_summary


#%load_ext autoreload
#%autoreload 2

IndentationError: expected an indented block after function definition on line 77 (player.py, line 82)

In [ ]:
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class GameManager:

    def __init__(self, 
                 policy_models, # network architectures
                 cfgs, # network configs
                 policies_weights, # trained network weights
                 board_shape_range, # board_shape_range
                 device,
                 force_finish_game=True):

        self.min_board_size, self.max_board_size = board_shape_range[0], board_shape_range[1]

        self.force_finish_game = force_finish_game

        self.device = device

        self.cfg1, self.cfg2 = cfgs[0], cfgs[1]
        self.policy_model1, self.policy_model2 = policy_models[0], policy_models[1]
        self.weights1, self.weights2 = policies_weights[0], policies_weights[1]

        ## instantiate the models:
        self.agent1 = self.policy_model1(self.cfg1)
        self.agent2 = self.policy_model2(self.cfg2)

        if self.weights1 != "random":
            self.agent1.load_state_dict(torch.load(self.weights1, weights_only=True, map_location=self.device))
        if self.weights2 != "random":
            self.agent2.load_state_dict(torch.load(self.weights2, weights_only=True, map_location=self.device))
    
        self.agent1.eval()
        self.agent2.eval()


    @property
    def print_agent_models(self, agent_id):
        if agent_id == 0:
            model_summary(self.agent1)
        else:
            model_summary(self.agent2)
            
    
    def _initiate_env(self, board_type, max_turns_per_game=9999):
        board_size = np.random.randint(self.min_board_size, self.max_board_size+1)
        board_config_dict = {"board_type":board_type, 
                             "n_players":2,
                             "board_size":(board_size, board_size)}
        new_env = EnvWrapper(board_config_dict,
                             [self.cfg1.tribe, self.cfg2.tribe],
                             max_turns_per_game=max_turns_per_game,
                             dense_reward=False)
        first_obs = new_env.reset()

        return new_env, first_obs


    def _run_env_step(self, env, obs, render=False):
        """
        Decides who plays based on the player_go_id
        """
        mask = env.get_action_mask()

        with torch.no_grad():
            if env.game.player_go_id == 0:
                action, all_probs, all_trajs,_, entropy, value = self.agent1(obs, mask)
            else:
                action, all_probs, all_trajs,_, entropy, value = self.agent2(obs, mask)

        obs, reward, done, info = env.step(action)
        if render:
            #env.render_with_trajs(shared_fog=True, critic_value=value,
            #                      action=action,
            #                      joint_probs=all_probs, traj_actions=all_trajs)
            env.render()

        return obs, reward, done, info, action, all_probs, all_trajs, entropy



    
    def run_games_and_collect_stats(self, n_games, max_turns):
        """
        Run all games and collect the stats
        """
        unit_collector = np.zeros((2, max_turns,2))
        win_collector = np.zeros(3)
        explore_collector = np.zeros((2, max_turns, n_games))
        entropy_collector = np.zeros((2, max_turns))
        
        for ng in tqdm(range(n_games)):
            
            env, obs = self._initiate_env(BoardType.Dummy)

            entropy_collection_flag = 1 ## start of the game is a first turn

            while env.game.turn < max_turns:

                obs, reward, done, info, action, all_probs, all_trajs, entropy = self._run_env_step(env, obs, render=False)

                if done:
                    win_collector[env.winner] += 1.0
                    break

                ## Collect unit creation:
                if action[0] == ActionTypes.CreateUnit:
                    unit_collector[env.game.player_go_id, env.game.turn, action[-1]] += 1.0 ## 0 for warr, 1 for rider 

                ## Collect introduced entropy:
                if entropy_collection_flag:
                    
                    if env.game.turn == max_turns:
                        break
                        
                    entropy_collector[env.game.player_go_id, env.game.turn] = entropy / float(len(env.game.players[env.game.player_go_id].units_under_control))
                    entropy_collection_flag = 0

                ## Collect exploration data:
                if action[0] == ActionTypes.EndTurn:
                    ## set collection flag for entropy:
                    entropy_collection_flag = 1

                    if env.game.turn == max_turns:
                        break
                    
                    ## WARNING: Here, player_go_id has already switched! So we need to collect from the other player
                    player_to_collect = env.game.players[(env.game.player_go_id + 1) % 2]
                    explore_collector[player_to_collect.player_id, env.game.turn, ng] = len(player_to_collect.uncovered_tile_ids)

            if done == False:
                win_collector[-1] += 1.0 ## game ended in a draw by timelimit

        unit_collector /= float(n_games)
        entropy_collector /= float(n_games)

        return unit_collector, win_collector, explore_collector, entropy_collector
            
                

            

In [ ]:
@dataclass
class TrainConfig:

    tribe      = Tribes.Omaji
    # ── Encoder ───────────────────────────────────────────────────────────────
    encoder_hidden_dim: int = 64
    encoder_n_heads:    int = 4
    encoder_depth:      int = 4

    # ── Selection heads ───────────────────────────────────────────────────────
    sel_n_heads:  int = 4
    sel_n_layers: int = 2

    # ── MLP ───────────────────────────────────────────────────────────────────
    mlp_hidden_dim: int = 128
    mlp_depth:      int = 3

    # ── Multi-scale convolutions ──────────────────────────────────────────────
    kernel_sizes:  Tuple[int, ...] = (5, 3)
    n_conv_layers: int             = 1

    # ── Movement context window ───────────────────────────────────────────────
    context_bias: int = 4



In [ ]:
@dataclass
class TrainConfig2:

    tribe      = Tribes.Imperius
    # ── Encoder ───────────────────────────────────────────────────────────────
    encoder_hidden_dim: int = 64
    encoder_n_heads:    int = 4
    encoder_depth:      int = 4

    # ── Selection heads ───────────────────────────────────────────────────────
    sel_n_heads:  int = 4
    sel_n_layers: int = 2

    # ── MLP ───────────────────────────────────────────────────────────────────
    mlp_hidden_dim: int = 128
    mlp_depth:      int = 3

    # ── Multi-scale convolutions ──────────────────────────────────────────────
    kernel_sizes:  Tuple[int, ...] = (5, 3)
    n_conv_layers: int             = 1

    # ── Movement context window ───────────────────────────────────────────────
    context_bias: int = 4


In [ ]:
cfgs = [TrainConfig(), TrainConfig2()]
models = [PolicyNetwork, PolicyNetwork]
weights = [
    "random",
    "random"
]

board_shape_range = (11,16)

In [ ]:
mngr1 = GameManager(models, cfgs, weights, board_shape_range, device)

In [ ]:
env, obs = mngr1._initiate_env(BoardType.Dummy)

In [ ]:


for i in range(100):
    obs, reward, done, info, action, all_probs, all_trajs, entropy = mngr1._run_env_step(env, obs, render=True)
    
    #for a, p in zip(all_trajs, all_probs):
    #    print(a, p)
    #print("Entropy value:", entropy)
    print(info)

    if done:
        break
    

In [ ]:
new_policy = mngr1.agent1

In [ ]:
a

## Run large scale stats simulations

In [ ]:
unit_collector, win_collector, explore_collector, entropy_collector = mngr1.run_games_and_collect_stats(n_games=500, max_turns=30)

In [ ]:
a

In [ ]:
import pickle

In [ ]:
save_dict = {
    "wins": win_collector,
    "units": unit_collector,
    "explore": explore_collector,
    "entropy": entropy_collector,
    "models": weights,
    "board_shape_range": board_shape_range,
}

In [ ]:
#with open(r"./eval/reward_curr_better_middle_vs_final_all_map_sizes", "wb") as handle:
#    pickle.dump(save_dict, handle)

In [ ]:
with open(r'./eval/win_rewards_middle_and_middle-1.pkl', mode='rb') as f:
    data = pickle.load(f)

In [ ]:
win_collector = data["wins"]
unit_collector = data["units"]
explore_collector = data["explore"]
entropy_collector = data["entropy"]

In [ ]:
data["board_shape_range"], data["models"]

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from scipy.stats import gaussian_kde

P0_DARK   = '#1a4fa8'
P0_LIGHT  = '#8fb3e8'   # lighter shade for text inside blue
P0_RIDER  = '#4a7fd4'   # mid-blue for stacked riders
P1_DARK   = '#c0392b'
P1_LIGHT  = '#e8918a'   # lighter shade for text inside red
P1_RIDER  = '#e06c63'   # mid-red for stacked riders
LABELS    = ['Player 0', 'Player 1']
COLORS    = [P0_DARK, P1_DARK]

def plot_speedometer(win_collector):
    """
    The drawable arc is scaled by the fraction of decisive games.
      - If all games are drawn  → needle is locked at 90° (up)
      - If no games are drawn   → full 180° arc available
      - Otherwise, the needle can only reach ±(decisive_fraction * 90°)
        from centre, then subdivided by P0/P1 win ratio within that range.

    Gradient drawn cleanly with imshow + semicircle clip mask (no moiré).
    Player labels and stats written in light colours inside the arc.
    """
    p0, p1, draws = int(win_collector[0]), int(win_collector[1]), int(win_collector[2])
    total          = p0 + p1 + draws
    decisive       = p0 + p1

    decisive_frac  = decisive / total if total > 0 else 0.0
    win_ratio      = p0 / decisive    if decisive > 0 else 0.5   # share going to P0

    # ── Needle angle ----------------------------------------------------------
    # Centre of arc = 90°.  Available half-sweep = decisive_frac * 90°.
    # P0 wins pull the needle left (toward 180°), P1 wins pull it right (0°).
    half_sweep   = decisive_frac * 90.0          # degrees either side of 90°
    needle_deg   = 90.0 + half_sweep * (2 * win_ratio - 1)
    # win_ratio=1  → +half_sweep (full left = P0 wins all)
    # win_ratio=0  → -half_sweep (full right = P1 wins all)
    # win_ratio=.5 → 0  (centre = equal)
    needle_rad   = np.deg2rad(needle_deg)

    # ── Figure & axes ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 5.2))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-0.48, 1.25)

    # ── Gradient fill — imshow rect clipped to semicircle ─────────────────────
    # Build a (H, W) RGBA image for the full [-1,1]×[0,1] region, then clip.
    res = 600
    xs  = np.linspace(-1, 1, res)
    ys  = np.linspace( 0, 1, res)
    XX, YY = np.meshgrid(xs, ys)

    # Angle of each pixel (0°=right, 180°=left), mapped to t ∈ [0,1] (left→right)
    T_img = (np.arctan2(YY, XX) / np.pi)          # ∈ [0, 1] across semicircle

    # Three-stop colormap:  blue → neutral grey → red
    cmap = LinearSegmentedColormap.from_list(
        'gauge',
        [(0.0, P1_DARK), (0.433, '#888888'), (0.566, '#888888'), (1.0, P0_DARK)],
        N=512,
    )
    rgba = cmap(T_img)                            # (H, W, 4)

    # Mask: keep only pixels inside semicircle (r ≤ 1, y ≥ 0)
    R      = np.sqrt(XX**2 + YY**2)
    inside = (R <= 1.0) & (YY >= 0)
    rgba[~inside, 3] = 0.0                        # transparent outside

    ax.imshow(
        rgba,
        origin='lower',
        extent=[-1, 1, 0, 1],
        aspect='auto',
        interpolation='bilinear',
        zorder=1,
    )

    # ── White tick marks on the arc ───────────────────────────────────────────
    """
    for deg in np.linspace(0, 180, 9):
        rad = np.deg2rad(deg)
        ax.plot(
            [0.76 * np.cos(rad), 1.01 * np.cos(rad)],
            [0.76 * np.sin(rad), 1.01 * np.sin(rad)],
            color='white', lw=1.6, alpha=0.85, zorder=4,
        )
    

    # ── Inner edge (white ring) ───────────────────────────────────────────────
    
    theta_ring = np.linspace(0, np.pi, 300)
    ax.plot(np.cos(theta_ring), np.sin(theta_ring),
            color='white', lw=2.0, zorder=4)
    ax.plot(0.76 * np.cos(theta_ring), 0.76 * np.sin(theta_ring),
            color='white', lw=2.0, zorder=4)
    # Baseline
    ax.plot([-1, 1], [0, 0], color='white', lw=2.0, zorder=4)
    """

    # ── Available-arc highlight (thin bright line showing the drawable range) ──
    lo_deg = 90.0 - half_sweep
    hi_deg = 90.0 + half_sweep
    arc_theta = np.linspace(np.deg2rad(lo_deg), np.deg2rad(hi_deg), 200)
    r_arc = 0.885
    #ax.plot(r_arc * np.cos(arc_theta), r_arc * np.sin(arc_theta),
    #        color='white', lw=3.5, alpha=0.55, zorder=3, solid_capstyle='round')

    # ── Needle ────────────────────────────────────────────────────────────────
    tip_x = 0.68 * np.cos(needle_rad)
    tip_y = 0.68 * np.sin(needle_rad)
    ax.annotate(
        "", xy=(tip_x, tip_y), xytext=(0, 0),
        arrowprops=dict(arrowstyle="-|>", color='#111111',
                        lw=3.0, mutation_scale=22),
        zorder=7,
    )
    ax.add_patch(plt.Circle((0, 0), 0.058, color='#111111', zorder=8))
    ax.add_patch(plt.Circle((0, 0), 0.032, color='white',   zorder=9))

    # ── Labels inside the coloured arc ───────────────────────────────────────
    def pct(n):
        return 100 * n / total if total > 0 else 0.0

    # Player labels at left and right ends of the arc
    #ax.text(-0.88, 0.38, "Player 0",
    #        ha='center', va='center', fontsize=12, fontweight='bold',
    #        color=P0_LIGHT, rotation=52, zorder=5)
    #ax.text( 0.88, 0.38, "Player 1",
    #        ha='center', va='center', fontsize=12, fontweight='bold',
    #        color=P1_LIGHT, rotation=-52, zorder=5)

    # Win/draw counts in lighter shades, sitting inside each side of the arc
    ax.text(-0.52, 0.20,
            f"{p0} wins\n({pct(p0):.1f}%)",
            ha='center', va='center', fontsize=9.5,
            color=P0_LIGHT, zorder=5)
    ax.text( 0.52, 0.20,
            f"{p1} wins\n({pct(p1):.1f}%)",
            ha='center', va='center', fontsize=9.5,
            color=P1_LIGHT, zorder=5)
    ax.text( 0.00, 0.60,
            f"Draws: {draws}\n({pct(draws):.1f}%)",
            ha='center', va='center', fontsize=9.5,
            color='#dddddd', zorder=5)

    ax.set_title("Win-Rate Gauge", fontsize=15, fontweight='bold', pad=6)
    plt.tight_layout()
    plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# B — Unit creation: stacked bars (warriors + riders) per turn, both players
# ══════════════════════════════════════════════════════════════════════════════

def plot_unit_histogram(unit_collector):
    """
    unit_collector : (2, 30, 2)  [player, turn, unit_type]
      unit_type 0 = warrior  (base bar)
      unit_type 1 = rider    (stacked on top, slightly hatched)

    Both players are shown side-by-side for each turn in a single plot.
    """
    turns = np.arange(30)
    bar_w = 0.38
    alpha = 0.78

    fig, ax = plt.subplots(figsize=(15, 5))

    for p in range(2):
        offset    = (p - 0.5) * bar_w
        warriors  = unit_collector[p, :, 0].astype(float)
        riders    = unit_collector[p, :, 1].astype(float)
        p_color   = COLORS[p]
        r_color   = [P0_RIDER, P1_RIDER][p]

        # Base: warriors
        ax.bar(
            turns + offset, warriors,
            width=bar_w,
            color=p_color,
            alpha=alpha,
            edgecolor='white',
            linewidth=0.5,
            label=f'{LABELS[p]} — Warriors',
            zorder=3,
        )
        # Stacked: riders (slightly lighter, hatched)
        ax.bar(
            turns + offset, riders,
            bottom=warriors,
            width=bar_w,
            color=r_color,
            alpha=alpha * 0.80,
            edgecolor='white',
            linewidth=0.5,
            hatch='//',
            label=f'{LABELS[p]} — Riders',
            zorder=3,
        )

    ax.set_title("Unit Creation per Turn — Both Players (stacked: Warriors + Riders)",
                 fontsize=13, fontweight='bold')
    ax.set_xlabel("Turn", fontsize=11)
    ax.set_ylabel("Units Created (total over games)", fontsize=11)
    ax.set_xticks(turns)
    ax.set_xticklabels(turns)
    ax.legend(fontsize=9.5, ncol=2)
    ax.grid(axis='y', alpha=0.28, linestyle='--')
    ax.set_facecolor('#f8f8f8')
    plt.tight_layout()
    plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# C — Policy entropy bar chart  (unchanged logic, kept for completeness)
# ══════════════════════════════════════════════════════════════════════════════

def plot_entropy(entropy_collector):
    turns = np.arange(30)
    bar_w = 0.35
    alpha = 0.80

    fig, ax = plt.subplots(figsize=(13, 5))

    for p in range(2):
        offset = (p - 0.5) * bar_w
        ax.bar(
            turns + offset,
            entropy_collector[p],
            width=bar_w,
            label=LABELS[p],
            color=COLORS[p],
            edgecolor='white',
            linewidth=0.6,
            alpha=alpha,
        )

    ax.set_title("Average entropy introduced per unit per turn", fontsize=15, fontweight='bold')
    ax.set_xlabel("Turn", fontsize=12)
    ax.set_ylabel("Entropy per unit", fontsize=12)
    ax.set_xticks(turns)
    ax.set_xticklabels(turns, fontsize=8)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.30, linestyle='--')
    ax.set_facecolor('#f8f8f8')
    plt.tight_layout()
    plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# D — Explore: 3D KDE surfaces  +  2D mean projection
# ══════════════════════════════════════════════════════════════════════════════

def plot_explore(explore_collector):
    """
    explore_collector : (2, 30, 500)
    - 0 values are filtered (placeholders).
    - 3D: one KDE surface per player.  P1 surface drawn as a wireframe /
      gridline mesh so both are visible when they overlap.
    - 2D: mean explore value per turn, bar chart with ±SE.
    """
    n_turns = explore_collector.shape[1]
    turns   = np.arange(n_turns)

    # ── Shared evaluation grid ────────────────────────────────────────────────
    # Filter zeros from the global range computation
    all_vals   = explore_collector[explore_collector > 0]
    val_min, val_max = all_vals.min(), all_vals.max()
    n_val_pts  = 80
    val_grid   = np.linspace(val_min, val_max, n_val_pts)

    # TURN × VALUE mesh for surface plot
    T_mesh, V_mesh = np.meshgrid(turns, val_grid, indexing='ij')   # (30, 80)

    # ── Build KDE surfaces ────────────────────────────────────────────────────
    Z = []   # one (30, n_val_pts) array per player
    for p in range(2):
        Zp = np.zeros((n_turns, n_val_pts), dtype=float)
        for t in range(n_turns):
            data = explore_collector[p, t]
            data = data[data > 0]                # drop placeholder zeros
            if data.size < 2:
                continue
            kde      = gaussian_kde(data, bw_method='silverman')
            Zp[t]    = kde(val_grid)
        Z.append(Zp)

    # ── 3D plot ───────────────────────────────────────────────────────────────
    fig3d = plt.figure(figsize=(14, 7))
    ax3d  = fig3d.add_subplot(111, projection='3d')

    # P0 — solid surface with low alpha
    surf0 = ax3d.plot_surface(
        T_mesh, V_mesh, Z[0],
        color=P0_DARK,
        alpha=0.55,
        linewidth=0,
        antialiased=True,
        zorder=2,
    )

    # P1 — wireframe / grid so it shows through P0
    ax3d.plot_wireframe(
        T_mesh, V_mesh, Z[1],
        color=P1_DARK,
        alpha=0.75,
        linewidth=0.6,
        rstride=2,
        cstride=4,
        zorder=3,
    )

    legend_handles = [
        mpatches.Patch(color=P0_DARK, alpha=0.7, label='Player 0 — solid surface'),
        mpatches.Patch(color=P1_DARK, alpha=0.7, label='Player 1 — wireframe'),
    ]
    ax3d.legend(handles=legend_handles, fontsize=10, loc='upper left')
    ax3d.set_xlabel("Turn",          fontsize=10, labelpad=10)
    ax3d.set_ylabel("Explore Value", fontsize=10, labelpad=10)
    ax3d.set_zlabel("KDE Density",   fontsize=10, labelpad=8)
    ax3d.set_xticks(np.arange(0, n_turns, 5))
    ax3d.set_title(
        "Exploration Value Distribution over Turns — KDE (3D)",
        fontsize=13, fontweight='bold',
    )
    ax3d.view_init(elev=30, azim=-55)
    plt.tight_layout()
    plt.show()

    # ── 2D projection: mean explore value per turn ────────────────────────────
    bar_w = 0.35
    fig2d, ax2d = plt.subplots(figsize=(13, 5))

    for p in range(2):
        mean_vals = []
        se_vals   = []
        for t in range(n_turns):
            data = explore_collector[p, t]
            data = data[data > 0]
            mean_vals.append(data.mean() if data.size > 0 else 0.0)
            se_vals.append(data.std() / np.sqrt(data.size) if data.size > 1 else 0.0)
        mean_vals = np.array(mean_vals)
        se_vals   = np.array(se_vals)

        offset = (p - 0.5) * bar_w
        ax2d.bar(
            turns + offset,
            mean_vals,
            width=bar_w,
            label=LABELS[p],
            color=COLORS[p],
            alpha=0.82,
            edgecolor='white',
            linewidth=0.6,
        )
        ax2d.errorbar(
            turns + offset,
            mean_vals,
            yerr=se_vals,
            fmt='none',
            ecolor='#333333',
            elinewidth=1.0,
            capsize=2,
        )

    ax2d.set_title(
        "Mean Exploration Value per Turn — 2D Projection (±SE, zeros excluded)",
        fontsize=13, fontweight='bold',
    )
    ax2d.set_xlabel("Turn",               fontsize=12)
    ax2d.set_ylabel("Mean Explore Value", fontsize=12)
    ax2d.set_xticks(turns)
    ax2d.set_xticklabels(turns, fontsize=8)
    ax2d.legend(fontsize=11)
    ax2d.grid(axis='y', alpha=0.30, linestyle='--')
    ax2d.set_facecolor('#f8f8f8')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_speedometer(win_collector)


In [ ]:
plot_unit_histogram(unit_collector)


In [ ]:
plot_entropy(entropy_collector)


In [ ]:
plot_explore(explore_collector)